In [15]:
import pandas as pd
df = pd.read_csv("/content/sample_data/country_wise_latest.csv")
df = df.drop(columns=[
    'Province/State', 'Lat', 'Long_',
    'Last Update', 'Incident_Rate', 'Case_Fatality_Ratio'
], errors='ignore')

print(df.head())

  Country/Region  Confirmed  Deaths  Recovered  Active  New cases  New deaths  \
0    Afghanistan      36263    1269      25198    9796        106          10   
1        Albania       4880     144       2745    1991        117           6   
2        Algeria      27973    1163      18837    7973        616           8   
3        Andorra        907      52        803      52         10           0   
4         Angola        950      41        242     667         18           1   

   New recovered  Deaths / 100 Cases  Recovered / 100 Cases  \
0             18                3.50                  69.49   
1             63                2.95                  56.25   
2            749                4.16                  67.34   
3              0                5.73                  88.53   
4              0                4.32                  25.47   

   Deaths / 100 Recovered  Confirmed last week  1 week change  \
0                    5.04                35526            737   
1   

# Data Cleaning & Preprocessing

In [16]:

print(df.isnull().sum())
df = df.fillna(0)
cols = ['Confirmed', 'Deaths', 'Recovered', 'Active']
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.drop_duplicates()

Country/Region            0
Confirmed                 0
Deaths                    0
Recovered                 0
Active                    0
New cases                 0
New deaths                0
New recovered             0
Deaths / 100 Cases        0
Recovered / 100 Cases     0
Deaths / 100 Recovered    0
Confirmed last week       0
1 week change             0
1 week % increase         0
WHO Region                0
dtype: int64


# Descriptive Statistics

In [17]:
total_confirmed = df['Confirmed'].sum()
total_deaths = df['Deaths'].sum()
total_recovered = df['Recovered'].sum()

print("Total Confirmed:", total_confirmed)
print("Total Deaths:", total_deaths)
print("Total Recovered:", total_recovered)

print(df.describe())

Total Confirmed: 16480485
Total Deaths: 654036
Total Recovered: 9468087
          Confirmed         Deaths     Recovered        Active     New cases  \
count  1.870000e+02     187.000000  1.870000e+02  1.870000e+02    187.000000   
mean   8.813094e+04    3497.518717  5.063148e+04  3.400194e+04   1222.957219   
std    3.833187e+05   14100.002482  1.901882e+05  2.133262e+05   5710.374790   
min    1.000000e+01       0.000000  0.000000e+00  0.000000e+00      0.000000   
25%    1.114000e+03      18.500000  6.265000e+02  1.415000e+02      4.000000   
50%    5.059000e+03     108.000000  2.815000e+03  1.600000e+03     49.000000   
75%    4.046050e+04     734.000000  2.260600e+04  9.149000e+03    419.500000   
max    4.290259e+06  148011.000000  1.846641e+06  2.816444e+06  56336.000000   

        New deaths  New recovered  Deaths / 100 Cases  Recovered / 100 Cases  \
count   187.000000     187.000000          187.000000             187.000000   
mean     28.957219     933.812834            3.

# Country-Based Analysis

In [18]:

top_countries = df.sort_values(by='Confirmed', ascending=False).head(10)
print(top_countries[['Country/Region', 'Confirmed', 'Deaths', 'Recovered']])

     Country/Region  Confirmed  Deaths  Recovered
173              US    4290259  148011    1325804
23           Brazil    2442375   87618    1846641
79            India    1480073   33408     951166
138          Russia     816680   13334     602249
154    South Africa     452529    7067     274925
111          Mexico     395489   44022     303810
132            Peru     389717   18418     272547
35            Chile     347923    9187     319954
177  United Kingdom     301708   45844       1437
81             Iran     293606   15912     255144


# Relationship Analysis

In [19]:

corr = df[['Confirmed', 'Deaths', 'Recovered', 'Active']].corr()
print(corr)

           Confirmed    Deaths  Recovered    Active
Confirmed   1.000000  0.934698   0.906377  0.927018
Deaths      0.934698  1.000000   0.832098  0.871586
Recovered   0.906377  0.832098   1.000000  0.682103
Active      0.927018  0.871586   0.682103  1.000000


# Dashboard

In [20]:
import plotly.express as px
import pandas as pd
!pip install dash
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)

countries = df['Country/Region'].unique()

app.layout = html.Div([
    html.H1("COVID-19 Dashboard", style={'textAlign': 'center'}),
    dcc.Dropdown(
        id='country',
        options=[{'label': c, 'value': c} for c in countries],
        value=countries[0]
    ),

    dcc.Graph(id='bar'),
    dcc.Graph(id='pie'),
    dcc.Graph(id='scatter'),
    dcc.Graph(id='heatmap')
])

# Callback

In [21]:
@app.callback(
    [Output('bar', 'figure'),
     Output('pie', 'figure'),
     Output('scatter', 'figure'),
     Output('heatmap', 'figure')],
    [Input('country', 'value')]
)
def update(selected_country):

    filtered = df[df['Country/Region'] == selected_country]

    top = df.sort_values(by='Confirmed', ascending=False).head(10)
    bar = px.bar(top, x='Country/Region', y='Confirmed',
                 title="Top Affected Countries")
    row = filtered.iloc[0]
    pie = px.pie(
        names=['Confirmed', 'Deaths', 'Recovered', 'Active'],
        values=[row['Confirmed'], row['Deaths'], row['Recovered'], row['Active']],
        title="Case Distribution"
    )
    scatter = px.scatter(df, x='Confirmed', y='Deaths',
                         title="Confirmed vs Deaths")
    heatmap = px.imshow(corr, text_auto=True,
                        title="Correlation Heatmap")

    return bar, pie, scatter, heatmap

In [22]:
if __name__ == '__main__':
    app.run(debug=True)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: on


In [23]:

high_risk = df.sort_values(by='Deaths', ascending=False).head(5)
print("High Risk Countries:\n", high_risk[['Country/Region', 'Deaths']])
df['Recovery Rate'] = df['Recovered'] / df['Confirmed'].replace(0, 1)
print("Average Recovery Rate:", df['Recovery Rate'].mean())

High Risk Countries:
      Country/Region  Deaths
173              US  148011
23           Brazil   87618
177  United Kingdom   45844
111          Mexico   44022
85            Italy   35112
Average Recovery Rate: 0.6482033679451074


# Line chart

In [24]:
top = df.sort_values(by='Confirmed', ascending=False).head(10)

fig_line = px.line(top,
                   x='Country/Region',
                   y='Confirmed',
                   markers=True,
                   title="Top Countries - Confirmed Cases")

fig_line.show()

# Bar chart

In [26]:
import plotly.express as px
top = df.sort_values(by='Confirmed', ascending=False).head(10)

fig_bar = px.bar(top,
                 x='Country/Region',
                 y='Confirmed',
                 title="Top 10 Affected Countries")

fig_bar.show()

# Pie chart

In [28]:
row = df.iloc[0]

fig_pie = px.pie(
    names=['Confirmed', 'Deaths', 'Recovered', 'Active'],
    values=[row['Confirmed'], row['Deaths'], row['Recovered'], row['Active']],
    title=f"Case Distribution - {row['Country/Region']}"
)

fig_pie.show()

# Scatter plot

In [31]:
fig_scatter = px.scatter(df,
                         x='Confirmed',
                         y='Deaths',
                         size='Confirmed',
                         hover_name='Country/Region',
                         title="Confirmed vs Deaths")

fig_scatter.show()

# Heatmap

In [32]:
corr = df[['Confirmed', 'Deaths', 'Recovered', 'Active']].corr()

fig_heatmap = px.imshow(corr,
                        text_auto=True,
                        title="Correlation Heatmap")

fig_heatmap.show()